# GRPO run book

A step-by-step walk through one GRPO step with the code in `grpo/`: load the data, build the prompt, sample a group of
answers, score them, compute advantages and log-probabilities, build the loss, and take one optimizer step. The last
sections show the four clip rules and a quick evaluation.

- It uses **SmolLM2-135M-Instruct** so it runs on a laptop (CPU or Apple MPS). Set `MODEL` to
  `Qwen/Qwen2.5-0.5B-Instruct` to match the experiment.
- Run it from the repo root, top to bottom. The first run downloads the model and GSM8K.
- The full training loop is `grpo/train.py`. This notebook is for seeing what each piece does.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch

from grpo.utils import get_device, seed_everything

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"  # the experiment uses "Qwen/Qwen2.5-0.5B-Instruct"
G = 4                 # answers per question (the experiment uses 8)
MAX_NEW_TOKENS = 256  # the experiment uses 512
EPS, BETA, LR = 0.2, 0.04, 1e-6

seed_everything(0)
device = get_device()
device

## 1. Data

GSM8K solutions end with `#### <number>`. `parse_ground_truth` keeps only that number, without commas.

In [ ]:
from grpo.data import load_gsm8k, parse_ground_truth

assert parse_ground_truth("some reasoning\n#### 72") == "72"
assert parse_ground_truth("some reasoning\n#### 1,200") == "1200"

train = load_gsm8k("train")
test = load_gsm8k("test")
print(len(train), "train questions,", len(test), "test questions")
train[0]

In [ ]:
import re

# every ground-truth answer should be a plain number
bad = [ex["answer"] for ex in train + test if not re.fullmatch(r"-?\d+(\.\d+)?", ex["answer"])]
len(bad), bad[:10]

## 2. Prompt

The system prompt asks for the final answer in `\boxed{}`, and `build_prompt` wraps it in the model's chat template.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from grpo.data import build_prompt

tok = AutoTokenizer.from_pretrained(MODEL)
question, answer = train[0]["question"], train[0]["answer"]
print(build_prompt(tok, question))

## 3. Sample a group

`sample_group` samples `G` answers at temperature 1.0. It returns the prompt ids, the completions, a mask of the real
tokens (up to and including the first end-of-sequence token), and which answers were cut off at `MAX_NEW_TOKENS`.

In [ ]:
from grpo.sampling import sample_group

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)
model.eval()

with torch.no_grad():
    prompt_ids, completions, mask, truncated = sample_group(model, tok, question, G, MAX_NEW_TOKENS)

print("completions:", tuple(completions.shape))
print("real tokens per answer:", mask.sum(dim=1).tolist())
print("truncated:", truncated.tolist())

texts = [tok.decode(completions[i][mask[i]], skip_special_tokens=True) for i in range(G)]
for i, text in enumerate(texts[:2]):
    print(f"\n--- answer {i} ---\n{text}")

## 4. Reward and advantages

The reward reads the first number inside the last `\boxed{}`: 0.1 if there is one, plus 1 if it matches the ground
truth. The advantage standardizes the rewards within the group, so if every answer gets the same reward, every
advantage is 0 and the group gives no gradient. If that happens here, run the sampling cell again.

In [ ]:
from grpo.loss import advantages
from grpo.reward import extract_answer, reward

rewards = torch.tensor([reward(t, answer) for t in texts], device=device)
adv = advantages(rewards)

print("ground truth:", answer)
print("extracted:   ", [extract_answer(t) for t in texts])
print("rewards:     ", rewards.tolist())
print("advantages:  ", [round(a, 3) for a in adv.tolist()])

## 5. Log-probabilities under the sampling policy and the reference

`token_logprobs` returns the log-probability of each completion token and, if asked, the entropy of the next-token
distribution. The sampling policy's log-probs are the denominator of the ratio $\rho$ (`old_lp`). The reference model
is a frozen copy of the starting model, used for the KL penalty.

In [ ]:
from grpo.sampling import token_logprobs

ref_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)
ref_model.eval()
ref_model.requires_grad_(False)

with torch.no_grad():
    old_lp, entropy = token_logprobs(model, prompt_ids, completions, return_entropy=True)
    ref_lp, _ = token_logprobs(ref_model, prompt_ids, completions)

m = mask.float()
print("log-probs:", tuple(old_lp.shape))
print("mean log-prob per real token:", [round(x, 3) for x in ((old_lp * m).sum(1) / m.sum(1)).tolist()])
print("mean entropy per real token (nats):", round(((entropy * m).sum() / m.sum()).item(), 3))

A sanity check: `token_logprobs` should match a plain `log_softmax` over the full sequence.

In [ ]:
with torch.no_grad():
    P = prompt_ids.shape[1]
    full = torch.cat([prompt_ids.repeat(G, 1), completions], dim=1)
    logp_all = torch.log_softmax(model(full).logits[:, P - 1 : -1].float(), dim=-1)
    plain = logp_all.gather(2, completions.unsqueeze(-1)).squeeze(-1)

print("max difference:", (old_lp - plain)[mask].abs().max().item())

## 6. The loss

Before the first update the policy still equals the sampling policy, so every ratio $\rho = \pi_\theta / \pi_{\text{old}}$
is 1 and nothing can be clipped. `grpo_loss` is the clipped surrogate minus $\beta$ times the per-token KL estimate,
averaged over each answer's real tokens and then over the group, and negated because the optimizer minimizes.

In [ ]:
from grpo.loss import grpo_loss

new_lp, _ = token_logprobs(model, prompt_ids, completions)  # with gradients this time
ratio = torch.exp(new_lp - old_lp)
print("max |rho - 1| before the update:", (ratio - 1)[mask].abs().max().item())

loss = grpo_loss(new_lp, old_lp, ref_lp, adv, mask, EPS, BETA, clip="symmetric")
print("loss:", loss.item())

## 7. One update

One AdamW step (weight decay 0, as in the experiment), with the gradient norm clipped to 1.0. Afterwards, answers with
a positive advantage should tend to become a little more likely, and answers with a negative advantage a little less
likely.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.0)
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()

with torch.no_grad():
    after_lp, _ = token_logprobs(model, prompt_ids, completions)

change = ((after_lp - old_lp) * m).sum(1) / m.sum(1)
print("grad norm (before clipping):", round(grad_norm.item(), 4))
print("max |rho - 1| after the update:", (torch.exp(after_lp - old_lp) - 1)[mask].abs().max().item())
print("advantage:                     ", [round(a, 3) for a in adv.tolist()])
print("mean log-prob change per token:", [round(c, 4) for c in change.tolist()])

## 8. The four clip rules

The experiment's arms differ only in `clip_bounds`, which turns each token's old probability $p$ into a lower and an
upper bound on $\rho$. These are the bounds for a few values of $p$, with each arm's settings from `configs/`:

In [ ]:
from grpo.loss import clip_bounds

arms = {  # name: (clip mode, eps, eps_high, c)
    "A symmetric":   ("symmetric", 0.2, 0.28, 1.0),
    "B Clip-Higher": ("decoupled", 0.2, 0.28, 1.0),
    "C linear":      ("prob_adaptive", 0.2, 0.28, 1.0),
    "D DCPO":        ("dcpo", 0.16, 0.2, 1.0),
}
ps = torch.tensor([0.01, 0.1, 0.5, 0.9, 1.0])

print(f"{'arm':15s}" + "".join(f"{f'p = {p:g}':>15s}" for p in ps.tolist()))
for name, (mode, eps, eps_high, c) in arms.items():
    lo, hi = clip_bounds(ps.log(), mode, eps, eps_high, c)
    lo, hi = torch.as_tensor(lo).expand_as(ps), torch.as_tensor(hi).expand_as(ps)
    print(f"{name:15s}" + "".join(f"{f'[{a:.2f}, {b:.2f}]':>15s}" for a, b in zip(lo.tolist(), hi.tolist())))

The experiment also counts how often each rule actually clips, split by the token's old probability. A token counts
as clipped when its ratio has moved past the bound in the direction its advantage pushes. After the single small
update above, with the symmetric rule:

In [ ]:
from grpo.loss import clip_counts

counts = clip_counts(after_lp, old_lp, adv, mask, "symmetric", EPS, 0.28, 1.0)
for bucket, (n, up, low) in counts.items():
    print(f"p_old {bucket:8s} {n:5d} tokens: {up} above the upper bound, {low} below the lower bound")

## 9. Quick evaluation

Greedy pass@1 on a few test questions, scored the same way the experiment scores all 1,319. On a laptop this takes a
few minutes.

In [ ]:
from grpo.eval import evaluate

metrics, samples = evaluate(model, tok, test[:16], MAX_NEW_TOKENS, batch_size=8)
metrics